# DSA 4060: Recommender Systems
## Lab Assignment 3: Content-Based Recommendation Engine

- **Student Name:** Nicholas Kinyanjui
- **Student ID:** 670178
- **Date:** October 2026
- **Topic:** Content-Based Filtering: Item Profiling, User Preference Modeling, and Vector Scoring

---

### Lab Overview & Objectives
Content-based recommender systems suggest items to users based on the descriptive attributes (features) of items the user has previously interacted with or rated highly. 

In this lab, we build an end-to-end content-based filtering pipeline from scratch using Python and `pandas`:
1. Represent items as structured feature vectors (Item Profiles).
2. Model user preferences by weighting item features according to user ratings.
3. Normalize the resulting user profile to create relative preference weights.
4. Calculate similarity / recommendation scores across all candidate items.
5. Filter out already-watched items and produce ranked Top-$N$ recommendations.
6. Analyze edge cases (e.g. single-rating user profiles), incorporate custom movies and users, and evaluate model limitations.

### Step 1 & 2: Import Libraries & Configure Environment
We import `pandas` for tabular data manipulation and vector operations, and `numpy` for numerical utilities.

In [1]:
import pandas as pd
import numpy as np

# Configure pandas display formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Pandas version: 3.0.6
NumPy version: 2.4.6


### Step 3: Create & Load the Movie Dataset
We load `movies.csv`, which contains 10 movies characterized across 5 distinct genres: **Action**, **Comedy**, **Drama**, **Romance**, and **SciFi**. Each genre is binary-encoded (1 if present, 0 if absent).

In [2]:
# Load dataset from movies.csv
movies = pd.read_csv("movies.csv")
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love at First Sight,0,1,0,1,0
2,3,Cosmic Odyssey,0,0,1,0,1
3,4,Laughter Therapy,0,1,0,0,0
4,5,Final Heist,1,0,1,0,0
5,6,Romantic Sunset,0,0,1,1,0
6,7,Cyberpunk 2099,1,0,0,0,1
7,8,Heartfelt Melodies,0,0,1,1,0
8,9,Funny Neighbors,0,1,0,1,0
9,10,Alien Encounter,1,0,0,0,1


### Step 4: Explore the Dataset
Let us examine the structure, dimensions, and statistical distribution of the dataset.

#### Question 1: What is the shape of the dataset and what do the binary values (1 and 0) represent?
- **Dataset Shape:** The dataset contains 10 rows and 7 columns.
- **Representation:** Each row represents a specific movie with a unique `movie_id` and `title`. The remaining 5 columns represent genre features. A value of **1** indicates that the movie belongs to that genre, while **0** indicates that the genre does not apply to that movie. This binary representation forms a one-hot / multi-hot feature matrix.

In [3]:
print("Dataset Shape:", movies.shape)
print("\nDataset Info:")
movies.info()
print("\nDataset Summary Statistics:")
movies.describe()

Dataset Shape: (10, 7)

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  10 non-null     int64
 1   title     10 non-null     str  
 2   Action    10 non-null     int64
 3   Comedy    10 non-null     int64
 4   Drama     10 non-null     int64
 5   Romance   10 non-null     int64
 6   SciFi     10 non-null     int64
dtypes: int64(6), str(1)
memory usage: 692.0 bytes

Dataset Summary Statistics:


,movie_id,Action,Comedy,Drama,Romance,SciFi
count,10.0000,10.0000,10.0000,10.0000,10.0000,10.0000
mean,5.5000,0.4000,0.3000,0.4000,0.4000,0.4000
std,3.0277,0.5164,0.4830,0.5164,0.5164,0.5164
min,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,3.2500,0.0000,0.0000,0.0000,0.0000,0.0000
50%,5.5000,0.0000,0.0000,0.0000,0.0000,0.0000
75%,7.7500,1.0000,0.7500,1.0000,1.0000,1.0000
max,10.0000,1.0000,1.0000,1.0000,1.0000,1.0000


### Step 5: Identify Structured Features
We define the feature subset used for modeling item representations.

#### Question 2: Why are these 5 features considered "structured"?
These features are considered **structured** because:
1. They adhere to a predefined, rigid schema where every column has a well-defined domain ($x \in \{0, 1\}$).
2. They are organized in a clean tabular matrix format, making them immediately amenable to direct mathematical operations (such as matrix multiplication, dot products, and vector addition) without requiring natural language processing (NLP) or parsing.
3. Unlike *unstructured* data (e.g., plot summaries, audio clips, video frames, user written reviews), structured features represent explicit categorical attributes.

In [4]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
print("Structured features identified:", features)

Structured features identified: ['Action', 'Comedy', 'Drama', 'Romance', 'SciFi']


### Step 6 & 7: Create and Inspect Item Profiles
The item profile is an $m \times k$ feature matrix where each movie $i$ is represented by a $k$-dimensional vector $\mathbf{v}_i = [v_{i1}, v_{i2}, \dots, v_{ik}]^T$.

#### Question 3: How does the item profile represent an item in vector space?
In vector space modeling, each movie is positioned as a point or vector in a $k$-dimensional Euclidean space (here $k=5$). The presence or absence of a genre determines the coordinate along that axis. Geometric distances or dot products between these vectors measure feature similarity between items.

In [5]:
movie_features = movies[features]
movie_features.index = movies["title"]
print("Item Profiles (first 5 movies):")
movie_features.head()

Item Profiles (first 5 movies):


,Action,Comedy,Drama,Romance,SciFi
title,,,,,
Galaxy Mission,1,0,0,0,1
Love at First Sight,0,1,0,1,0
Cosmic Odyssey,0,0,1,0,1
Laughter Therapy,0,1,0,0,0
Final Heist,1,0,1,0,0


### Step 8: Create User Ratings (User A)
We now define explicit rating feedback from **User A**, who has watched and rated four movies on a scale of 1 to 5 stars.
- Movie 1 (*Galaxy Mission*): 5 stars
- Movie 3 (*Cosmic Odyssey*): 4 stars
- Movie 5 (*Final Heist*): 4 stars
- Movie 7 (*Cyberpunk 2099*): 5 stars

In [6]:
ratings = pd.DataFrame({
    "movie_id": [1, 3, 5, 7],
    "rating": [5, 4, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,3,4
2,5,4
3,7,5


### Step 9 & 10: Merge Ratings with Movie Profiles and Extract Rated Features
To understand what features User A enjoyed, we merge the ratings with the movie metadata.

In [7]:
# Step 9: Merge user ratings with movies
user_movies = ratings.merge(movies, on="movie_id")
print("User A's Rated Movies:")
user_movies[["movie_id", "title", "rating"] + features]

User A's Rated Movies:


,movie_id,title,rating,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,5,1,0,0,0,1
1,3,Cosmic Odyssey,4,0,0,1,0,1
2,5,Final Heist,4,1,0,1,0,0
3,7,Cyberpunk 2099,5,1,0,0,0,1


In [8]:
# Step 10: Extract rated features
rated_features = user_movies[features]
rated_features.index = user_movies["title"]
rated_features

,Action,Comedy,Drama,Romance,SciFi
title,,,,,
Galaxy Mission,1,0,0,0,1
Cosmic Odyssey,0,0,1,0,1
Final Heist,1,0,1,0,0
Cyberpunk 2099,1,0,0,0,1


### Step 11 & 12: Weight Features by Ratings
We multiply each movie's feature vector by the rating given by the user:
$$\mathbf{W}_i = r_i \times \mathbf{v}_i$$

**Interpretation:** A highly rated movie contributes more strongly to the user's preference profile. If a user awards 5 stars to a SciFi movie, the SciFi dimension receives a weight of 5. If they give 1 star, it contributes only 1.

In [9]:
weighted_features = rated_features.mul(
    user_movies["rating"].values,
    axis=0
)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
title,,,,,
Galaxy Mission,5,0,0,0,5
Cosmic Odyssey,0,0,4,0,4
Final Heist,4,0,4,0,0
Cyberpunk 2099,5,0,0,0,5


### Step 13: Build the User Profile
We sum the weighted features across all movies watched by the user:
$$\mathbf{u}_{raw} = \sum_{i \in \mathcal{I}_{watched}} r_i \cdot \mathbf{v}_i$$
The resulting scores summarize which features the user tends to prefer overall.

In [10]:
user_profile = weighted_features.sum()
user_profile

Action     14
Comedy      0
Drama       8
Romance     0
SciFi      14
dtype: int64

### Step 14: Interpret the User Profile
We rank the raw genre preference scores in descending order.

#### Question: Which two genres appear strongest for this user? Explain using the profile scores.
- **Top Two Strongest Genres:** **Action** (score: 14) and **SciFi** (score: 14).
- **Explanation:** 
  - User A watched *Galaxy Mission* (rated 5, Action=1, SciFi=1), contributing 5 to Action and 5 to SciFi.
  - User A watched *Cosmic Odyssey* (rated 4, Drama=1, SciFi=1), contributing 4 to Drama and 4 to SciFi.
  - User A watched *Final Heist* (rated 4, Action=1, Drama=1), contributing 4 to Action and 4 to Drama.
  - User A watched *Cyberpunk 2099* (rated 5, Action=1, SciFi=1), contributing 5 to Action and 5 to SciFi.
  - Total Action score = $5 + 4 + 5 = 14$.
  - Total SciFi score = $5 + 4 + 5 = 14$.
  - Both Action and SciFi are tied at 14 points, making them co-dominant preferences. Drama follows with 8 points, while Comedy and Romance received 0 points because User A watched zero movies with these genres.

In [11]:
user_profile.sort_values(
    ascending=False
)

Action     14
SciFi      14
Drama       8
Comedy      0
Romance     0
dtype: int64

### Step 15: Normalize the User Profile
We normalize the user profile by dividing by the sum of all profile scores:
$$\mathbf{u}_{norm} = \frac{\mathbf{u}_{raw}}{\sum_j u_{raw, j}}$$

**Why Normalize?**
1. Normalization converts raw feature sums into **proportions** (relative weights) that sum to 1.0.
2. It eliminates magnitude distortions caused by the number of ratings (e.g. an active user who rated 50 movies vs. a user who rated 3 movies).
3. It makes the scores directly comparable and interpretable as percentage affinities.

In [12]:
user_profile_normalized = (
    user_profile / user_profile.sum()
)
user_profile_normalized

Action    0.3889
Comedy    0.0000
Drama     0.2222
Romance   0.0000
SciFi     0.3889
dtype: float64

### Step 16: Score Every Movie
We compute the dot product between every movie's feature vector and the normalized user profile vector:
$$\text{Score}_i = \mathbf{v}_i \cdot \mathbf{u}_{norm} = \sum_{j} v_{ij} \times u_{norm, j}$$

In [13]:
movies["recommendation_score"] = (
    movie_features.reset_index(drop=True)
    .mul(user_profile_normalized)
    .sum(axis=1)
)
movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.7778
1,Love at First Sight,0.0000
2,Cosmic Odyssey,0.6111
3,Laughter Therapy,0.0000
4,Final Heist,0.6111
5,Romantic Sunset,0.2222
6,Cyberpunk 2099,0.7778
7,Heartfelt Melodies,0.2222
8,Funny Neighbors,0.0000
9,Alien Encounter,0.7778


### Step 17: Rank the Movies
We sort the movies by recommendation score in descending order.

**Interpretation:** A higher recommendation score means the movie contains more of the features preferred by the user.

In [14]:
ranked_movies = movies.sort_values(
    "recommendation_score",
    ascending=False
)
ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.7778
9,Alien Encounter,0.7778
6,Cyberpunk 2099,0.7778
2,Cosmic Odyssey,0.6111
4,Final Heist,0.6111
7,Heartfelt Melodies,0.2222
5,Romantic Sunset,0.2222
1,Love at First Sight,0.0000
3,Laughter Therapy,0.0000
8,Funny Neighbors,0.0000


### Step 18: Remove Movies Already Watched
In recommender systems, recommending items the user has already consumed degrades user experience. We filter out the watched movie IDs.

In [15]:
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[
    ~ranked_movies["movie_id"].isin(watched_movies)
]
recommendations[["title", "recommendation_score"]]

,title,recommendation_score
9,Alien Encounter,0.7778
7,Heartfelt Melodies,0.2222
5,Romantic Sunset,0.2222
1,Love at First Sight,0.0000
3,Laughter Therapy,0.0000
8,Funny Neighbors,0.0000


### Step 19: Generate Top 3 Recommendations
We extract the Top 3 unwatched candidates for User A.

In [16]:
top_3 = recommendations[
    ["title", "recommendation_score"]
].head(3)
top_3

,title,recommendation_score
9,Alien Encounter,0.7778
7,Heartfelt Melodies,0.2222
5,Romantic Sunset,0.2222


### Step 20: Explain the Recommendations
For each of the Top 3 movies, write one sentence explaining why the model recommended it referring to the user's strongest feature preferences:

1. **Alien Encounter (Score: 0.7778):** Recommended because it features both **Action** and **SciFi**, perfectly matching User A's two strongest preference genres which account for over 77% of their total profile weight.
2. **Romantic Sunset (Score: 0.2222):** Recommended because it contains **Drama**, which corresponds to User A's third preferred genre (22.22% preference weight), despite having no romance affinity.
3. **Heartfelt Melodies (Score: 0.2222):** Recommended because it includes **Drama**, matching User A's positive past ratings on dramatic storylines such as *Cosmic Odyssey* and *Final Heist*.

### Step 21: Build a Reusable Function
We encapsulate the full content-based filtering pipeline into a reusable Python function `recommend_movies`.

In [17]:
def recommend_movies(movies, ratings, features, n=3):
    user_movies = ratings.merge(movies, on="movie_id")
    rated_features = user_movies[features]
    weighted_features = rated_features.mul(
        user_movies["rating"],
        axis=0
    )
    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()
    scores = (
        movies[features]
        .mul(user_profile)
        .sum(axis=1)
    )
    results = movies.copy()
    results["recommendation_score"] = scores
    watched = ratings["movie_id"]
    results = results[
        ~results["movie_id"].isin(watched)
    ]
    return results.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

# Test the function on User A
recommend_movies(
    movies,
    ratings,
    features
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Alien Encounter,1,0,0,0,1,0.7778
5,6,Romantic Sunset,0,0,1,1,0,0.2222
7,8,Heartfelt Melodies,0,0,1,1,0,0.2222


### Step 22: Test Another User (User B)
We test the recommender on a second user with distinct tastes:
- Movie 2 (*Love at First Sight*): 5 stars (Comedy=1, Romance=1)
- Movie 4 (*Laughter Therapy*): 4 stars (Comedy=1)
- Movie 6 (*Romantic Sunset*): 5 stars (Drama=1, Romance=1)
- Movie 9 (*Funny Neighbors*): 4 stars (Comedy=1, Romance=1)

#### Comparison: Did User A and User B receive the same recommendations? Explain why or why not.
- **Result:** No, User A and User B received **completely different** recommendations.
- **Why?** Content-based filtering is driven entirely by individual preference profiles:
  - User A's profile was dominated by **SciFi (38.89%)** and **Action (38.89%)**, leading to Top recommendation *Alien Encounter* (Action + SciFi).
  - User B's profile is dominated by **Romance (43.75%)** and **Comedy (40.62%)** with secondary **Drama (15.62%)**, leading to Top recommendation *Heartfelt Melodies* (Drama + Romance, score: 0.5938).
  - This demonstrates that the algorithm adapts effectively to divergent individual tastes.

In [18]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

print("User B Top 3 Recommendations:")
recommend_movies(
    movies,
    user_b_ratings,
    features
)

User B Top 3 Recommendations:


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Heartfelt Melodies,0,0,1,1,0,0.5938
2,3,Cosmic Odyssey,0,0,1,0,1,0.1562
4,5,Final Heist,1,0,1,0,0,0.1562


### Step 23: Thinking Exercise (User C)
Suppose **User C** has watched only *Galaxy Mission* and rated it 5 stars.

#### 4. What will the model learn about User C?
The model will learn that User C has a profile derived solely from *Galaxy Mission*. Because *Galaxy Mission* has features `Action=1` and `SciFi=1`, the model infers that User C is interested exclusively in Action and SciFi, assigning zero preference to Comedy, Drama, and Romance.

#### 5. Which genres will dominate the profile?
**Action** and **SciFi** will completely dominate the profile with equal weight:
- Raw Profile: Action = 5, SciFi = 5, Comedy = 0, Drama = 0, Romance = 0.
- Normalized Profile: Action = 0.50 (50%), SciFi = 0.50 (50%).

#### 6. What could go wrong with the recommendations?
1. **Extreme Overspecialization (Filter Bubble):** The system will only recommend movies containing Action or SciFi, completely blocking all other genres even though the user might enjoy comedy or drama.
2. **False Generalization:** The user may have liked *Galaxy Mission* for its lead actor, soundtrack, visual effects, or humor, rather than simply because it is an Action/SciFi movie. The model attributes 100% of the rating to genre flags.
3. **Redundancy:** Recommendations will be nearly identical in theme to the single movie already watched.

#### 7. Is one rating enough to understand a user's preferences?
**No.** A single rating exhibits high statistical variance and low sample representativeness. Preferences are multi-dimensional, context-dependent (e.g., mood, social setting, time of day), and cannot be accurately estimated from a single observation without suffering from severe cold-start uncertainty.

In [19]:
# Demonstrate User C mathematically
user_c_ratings = pd.DataFrame({
    "movie_id": [1],
    "rating": [5]
})

print("User C Recommendations:")
recommend_movies(
    movies,
    user_c_ratings,
    features,
    n=4
)

User C Recommendations:


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Alien Encounter,1,0,0,0,1,1.0000
6,7,Cyberpunk 2099,1,0,0,0,1,1.0000
2,3,Cosmic Odyssey,0,0,1,0,1,0.5000
4,5,Final Heist,1,0,1,0,0,0.5000


### Step 24: Add Your Unique Movie
Every student must add one unique movie with their own title and feature combination.
Here Nicholas Kinyanjui adds:
- `movie_id`: 11
- `title`: *"Nairobi Nights: Urban Pulse"*
- Genre Features: Comedy = 1, Drama = 1 (Action=0, Romance=0, SciFi=0)

In [20]:
new_movie = {
    "movie_id": 11,
    "title": "Nairobi Nights: Urban Pulse",
    "Action": 0,
    "Comedy": 1,
    "Drama": 1,
    "Romance": 0,
    "SciFi": 0
}

movies = pd.concat(
    [movies, pd.DataFrame([new_movie])],
    ignore_index=True
)

print("Updated Movies Catalog (last 3 rows):")
movies.tail(3)

Updated Movies Catalog (last 3 rows):


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
8,9,Funny Neighbors,0,1,0,1,0,0.0000
9,10,Alien Encounter,1,0,0,0,1,0.7778
10,11,Nairobi Nights: Urban Pulse,0,1,1,0,0,NaN


### Step 25: Create Your Own User (Nicholas Kinyanjui)
We define ratings for Nicholas across four distinct movies:
- Movie 1 (*Galaxy Mission*): 5 stars
- Movie 3 (*Cosmic Odyssey*): 4 stars
- Movie 6 (*Romantic Sunset*): 2 stars (disliked romantic drama)
- Movie 9 (*Funny Neighbors*): 3 stars (neutral comedy romance)

We generate Top 5 recommendations for this personalized profile.

In [21]:
my_user = pd.DataFrame({
    "movie_id": [1, 3, 6, 9],
    "rating": [5, 4, 2, 3]
})

print("Nicholas Kinyanjui's Top 5 Recommendations:")
recommend_movies(
    movies,
    my_user,
    features,
    n=5
)

Nicholas Kinyanjui's Top 5 Recommendations:


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
6,7,Cyberpunk 2099,1,0,0,0,1,0.5000
9,10,Alien Encounter,1,0,0,0,1,0.5000
4,5,Final Heist,1,0,1,0,0,0.3929
7,8,Heartfelt Melodies,0,0,1,1,0,0.3929
10,11,Nairobi Nights: Urban Pulse,0,1,1,0,0,0.3214


### Step 26: Interpretation Questions

Let us inspect Nicholas's profile mathematically to answer the interpretation questions:

In [22]:
my_user_movies = my_user.merge(movies, on="movie_id")
my_weighted = my_user_movies[features].mul(my_user_movies["rating"], axis=0)
my_profile = my_weighted.sum()
my_profile_norm = my_profile / my_profile.sum()

print("Nicholas's Raw Profile Scores:")
print(my_profile.sort_values(ascending=False))
print("\nNicholas's Normalized Profile Proportions:")
print(my_profile_norm.sort_values(ascending=False))

Nicholas's Raw Profile Scores:
SciFi      9
Drama      6
Action     5
Romance    5
Comedy     3
dtype: int64

Nicholas's Normalized Profile Proportions:
SciFi     0.3214
Drama     0.2143
Action    0.1786
Romance   0.1786
Comedy    0.1071
dtype: float64


#### 8. What are your user's two strongest preferences?
Based on the computed normalized profile:
1. **SciFi:** 0.3214 (32.14% of total weight, raw score: 9)
2. **Drama:** 0.2143 (21.43% of total weight, raw score: 6)
*Action* and *Romance* tie for third at 0.1786 (17.86% each), followed by *Comedy* at 0.1071 (10.71%).

#### 9. Which three movies received the highest recommendation scores?
The Top 3 recommendations produced by `recommend_movies` are:
1. **Cyberpunk 2099** (Score: **0.5000**)
2. **Alien Encounter** (Score: **0.5000**)
3. **Final Heist** (Score: **0.3929**) *(tied with Heartfelt Melodies at 0.3929)*

#### 10. Why did the top movie rank first?
*Cyberpunk 2099* has features `Action=1` and `SciFi=1`. SciFi is the user's primary preference ($0.3214$) and Action is a strong secondary preference ($0.1786$). Summing these gives:
$$\text{Score} = 0.3214 + 0.1786 = 0.5000$$
This matches the highest possible affinity score for unwatched candidates in the catalog.

#### 11. Did any recommendation surprise you?
**Yes.** *Heartfelt Melodies* (Drama=1, Romance=1) achieved a score of **0.3929**, tying for 3rd place, despite the user having awarded only 2 stars to *Romantic Sunset* (which also contains Drama and Romance). 
This occurred because the current algorithm sums positive rating weights without penalizing low ratings (ratings $\le 2$ still added positive points to Drama and Romance rather than subtracting). Additionally, the user rated *Cosmic Odyssey* 4 stars, which boosted the Drama weight significantly.

#### 12. What important information is the model ignoring?
1. **Negative Sentiment / Rating Penalties:** A 1 or 2-star rating still contributes positively to the user profile instead of acting as a negative preference signal.
2. **Rich Metadata Attributes:** The model ignores directors, leading actors/actresses, production studio, release year, language, mood/tone, and content maturity ratings.
3. **Implicit Signals & Context:** The model ignores watch completion rate, rewatch behavior, time of day/season, device type, and temporal recency (recent ratings vs. ratings from years ago).

### Step 27: Limitations of this Content-Based Recommender
We evaluate key systemic weaknesses in this recommendation architecture:

1. **Coarse Feature Representation (Low Expressiveness):**
   Only five binary genre features are utilized. Binary flags cannot capture subtle genre blends, artistic nuance, directing style, pacing, or cinematic tone. A high-budget sci-fi thriller is treated as identical in feature space to a low-budget sci-fi comedy.
   
2. **Overspecialization and the Filter Bubble (Serendipity Deficit):**
   Content-based filtering strictly recommends items that are similar to what the user has previously consumed. The model cannot introduce users to novel genres or serendipitous discoveries outside their established historical profile.
   
3. **Cold-Start Problem for New Users:**
   When a new user joins the platform with no viewing or rating history, $\mathbf{u}_{raw} = \mathbf{0}$, meaning no personalized recommendation score can be computed without asking explicit onboarding questions.
   
4. **Metadata Quality and Hand-Crafting Dependency:**
   The algorithm relies entirely on accurate, comprehensive metadata. If items have missing, noisy, or subjective genre tags, recommendation accuracy collapses.
   
5. **Lack of Collaborative / Community Wisdom:**
   Unlike Collaborative Filtering, this model ignores community ratings and behavior from other users. It cannot leverage the principle that users with similar tastes can discover items across different genres.

### Step 28: Content-Based Filtering Workflow & Conclusion

#### Summary Workflow
The end-to-end architecture implemented in this lab follows the standard content-based filtering pipeline:

```text
┌─────────────────┐     ┌────────────────┐     ┌──────────────┐     ┌──────────────┐
│  Item Metadata  │ ──> │ Item Profiles  │ ──> │ User Ratings │ ──> │ User Profile │
└─────────────────┘     └────────────────┘     └──────────────┘     └──────────────┘
                                                                           │
┌─────────────────────────┐     ┌──────────────┐     ┌──────────────┐      │
│ Top-N Recommendations   │ <── │ Remove Seen  │ <── │ Score & Rank │ <────┘
└─────────────────────────┘     └──────────────┘     └──────────────┘
```

#### Final Conclusion
In this lab, we successfully built, evaluated, and interpreted a content-based recommendation engine for **Nicholas Kinyanjui (670178)**. We demonstrated how user ratings transform item feature vectors into a personalized preference distribution, tested behavioral variations across multiple user profiles, analyzed the single-rating edge case, and critically identified the structural limitations of binary genre-based scoring.